# Critic v2 — ResNet-18 from scratch (по статье PA-RL)

**Что меняется vs предыдущих попыток:**
- Эмбеддинги от SmolVLA **не используются** для critic'а. Они хороши для policy action prediction, плохи для оценки Q.
- Вместо них **отдельный ResNet-18 + State MLP** — обучается с нуля на TD loss.
- **Twin Q** (2 critics, target = min) против overestimation.
- **Random shift augmentation 4 px** — стандартный приём из DrQ-v2 / PA-RL.

**Architecture (из статьи Appendix B.1):**
```
img1 (256x256x3) → ResNet18(out=512)  ─┐
img2 (256x256x3) → ResNet18(out=512)  ─┼─ concat → MLP(512,512,512) → Q
state (8d)       → MLP(64)            ─┤   с action concat на каждом слое
action (7d)      → ─────────────────────┘
```

**Запуск:**
1. Pretrain critic ~50k шагов на 50k frames LIBERO-spatial
2. Q-select эвал на 13 init_states
3. Сравним с baseline (8/13) и v1 critic (8/13)

## 1. Setup

In [ ]:
import os
os.environ["MUJOCO_GL"]="egl"; os.environ["PYOPENGL_PLATFORM"]="egl"

import sys, math, time, gc, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.functional as TF

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
device = torch.device("cuda:0")
torch.backends.cudnn.benchmark = True

MODEL_ID    = "HuggingFaceVLA/smolvla_libero"
CRITIC_PATH = "/workspace/out/critic_resnet.pt"

TASK_DESCRIPTION = "pick up the black bowl between the plate and the ramekin and place it on the plate"

TASK_SUITE_NAME       = "libero_spatial"
TASK_ID               = 0
ENV_IMAGE_SIZE        = 256
SEED                  = 42
MAX_STEPS             = 90
NUM_STABILIZATION_STEPS = 10
INIT_STATES_IDS       = [1, 2, 6, 7, 13, 22, 23, 27, 32, 35, 38, 47, 46] # не менять!

torch.manual_seed(SEED); np.random.seed(SEED)

from huggingface_hub import login
login(token="", add_to_git_credential=False)
print("[hf] logged in")


## 2. Image preprocessing с кэшем

Загружаем все изображения LIBERO-spatial в numpy кэш, downscale до 96×96 (для скорости critic'а).
Хранятся uint8 → ~50000 × 96 × 96 × 3 × 2 cameras = ~2.6 GB. Влезает в RAM.

In [ ]:
from datasets import load_dataset
ds = load_dataset("k1000dai/libero-spatial")["train"]
print(f"Dataset size: {len(ds)}")

# Кэш в исходном разрешении 256×256 uint8 (~20 GB на диск, столько же в RAM).
# downscale до CRITIC_IMG_SIZE делается на лету на GPU вместе с random shift —
# random shift применяется к ПОЛНОМУ разрешению (8 px на 256 = эквивалент DrQ-v2
# pad=4 на working resolution 128, то есть `4 pixels` из App B.1 PA-RL).
# 128×128 (вместо стандартных 84×84) — для сохранения видимости мелких объектов
# LIBERO (bowl, plate, ramekin), которые при 84×84 теряют до 60% детализации.
RAW_IMG_SIZE   = 256
CRITIC_IMG_SIZE = 128  # было 84: для LIBERO с маленькими объектами
IMG_CACHE_PATH = "/workspace/data/img_cache_critic_256.npz"

if os.path.exists(IMG_CACHE_PATH):
    print(f"Загружаем image cache из {IMG_CACHE_PATH}")
    data = np.load(IMG_CACHE_PATH)
    img1_cache = data["img1"]  # uint8 [N, 96, 96, 3]
    img2_cache = data["img2"]
    state_cache = data["state"].astype(np.float32)
    action_cache = data["action"].astype(np.float32)
    episode_cache = data["episode"].astype(np.int64)
    frame_cache = data["frame"].astype(np.int64)
    print(f"Cache loaded: {img1_cache.shape}")
else:
    print("Строим image cache (один раз, ~3-5 мин)…")
    N = len(ds)
    img1_cache = np.zeros((N, RAW_IMG_SIZE, RAW_IMG_SIZE, 3), dtype=np.uint8)
    img2_cache = np.zeros((N, RAW_IMG_SIZE, RAW_IMG_SIZE, 3), dtype=np.uint8)
    state_cache = np.zeros((N, 8), dtype=np.float32)
    action_cache = np.zeros((N, 7), dtype=np.float32)
    episode_cache = np.zeros(N, dtype=np.int64)
    frame_cache = np.zeros(N, dtype=np.int64)
    
    BATCH = 256
    t0 = time.time()
    for s in range(0, N, BATCH):
        e = min(s + BATCH, N)
        rows = ds[s:e]
        for i, (im1, im2, st, ac, ep, fr) in enumerate(zip(
            rows["observation.images.image"],
            rows["observation.images.wrist_image"],
            rows["observation.state"],
            rows["action"],
            rows["episode_index"],
            rows["frame_index"],
        )):
            # Сохраняем в исходном разрешении — без потери информации
            img1_cache[s+i] = np.array(im1, dtype=np.uint8)
            img2_cache[s+i] = np.array(im2, dtype=np.uint8)
            state_cache[s+i] = np.array(st, dtype=np.float32)
            action_cache[s+i] = np.array(ac, dtype=np.float32)[:7]
            episode_cache[s+i] = ep
            frame_cache[s+i] = fr
        if s % (BATCH * 20) == 0:
            print(f"  {s}/{N} ({100*s/N:.0f}%, {(time.time()-t0):.0f}s)")
    
    print(f"Сохраняем cache в {IMG_CACHE_PATH}…")
    os.makedirs(os.path.dirname(IMG_CACHE_PATH), exist_ok=True)
    np.savez(IMG_CACHE_PATH,
             img1=img1_cache, img2=img2_cache,
             state=state_cache, action=action_cache,
             episode=episode_cache, frame=frame_cache)
    print(f"Готово, {(time.time()-t0)/60:.1f} мин")

print(f"\nCache stats:")
print(f"  img1: {img1_cache.shape} {img1_cache.dtype} ({img1_cache.nbytes/1e9:.2f} GB)")
print(f"  img2: {img2_cache.shape}")
print(f"  state: {state_cache.shape} range [{state_cache.min():.2f}, {state_cache.max():.2f}]")
print(f"  action: {action_cache.shape} range [{action_cache.min():.2f}, {action_cache.max():.2f}]")


## 3. Нормализация state

Обычно SmolVLA preprocessor нормализует state. Мы делаем то же — нормализуем по mean/std из datasets.

In [ ]:
state_mean = state_cache.mean(axis=0)
state_std  = state_cache.std(axis=0) + 1e-6
state_norm_cache = (state_cache - state_mean) / state_std
print(f"State mean: {state_mean}")
print(f"State std:  {state_std}")
print(f"State norm range: [{state_norm_cache.min():.2f}, {state_norm_cache.max():.2f}]")


## 3.5. Сбор failure-data через rollouts baseline policy

**Зачем это нужно:** датасет `k1000dai/libero-spatial` содержит **только успешные демо-эпизоды**. С reward shaping `−1 / 0` это означает, что Q-target = `−(steps remaining to terminal)`, и Q учится аппроксимировать `V(s)` без зависимости от action — потому что любая (s, a) пара из датасета ведёт в terminal с reward=0 примерно через одинаковое число шагов.

Поэтому local refine PA-RL не работает: `∇_a Q ≈ 0` в окрестности demo, критик не различает близкие к demo действия.

**Решение:** прогнать baseline SmolVLA в env-е, собрать смесь успешных и неудачных rollouts, добавить в буфер. Failure-эпизоды получают `done=1, reward=−1` на терминале (timeout), success-эпизоды — `done=1, reward=0`. Это даёт Q-функции **реальный TD-сигнал**, что одни (s,a) ведут к успеху, другие нет.


In [ ]:
# ── Загрузка SmolVLA + env только для сбора rollouts ──
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

policy = SmolVLAPolicy.from_pretrained(MODEL_ID).to(device).eval()
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=MODEL_ID,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)
print(f"SmolVLA loaded for rollout collection")

import builtins
builtins.input = lambda _: "n"
os.environ["LIBERO_DATA_PATH"] = "/workspace/libero_data"
os.makedirs("/workspace/libero_data", exist_ok=True)

from libero.libero import get_libero_path, benchmark
from libero.libero.envs import OffScreenRenderEnv

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[TASK_SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
task_description_libero = task.language
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
env = OffScreenRenderEnv(bddl_file_name=task_bddl_file,
                         camera_heights=ENV_IMAGE_SIZE, camera_widths=ENV_IMAGE_SIZE)
env.seed(SEED)
init_states = task_suite.get_task_init_states(TASK_ID)
print(f"Task: {task_description_libero}")


def quat2axisangle(quat):
    quat = quat.astype(np.float32).copy()
    quat[3] = np.clip(quat[3], -1.0, 1.0)
    den = np.sqrt(max(1e-12, 1.0 - quat[3]**2))
    if np.isclose(den, 0.0): return np.zeros(3, dtype=np.float32)
    angle = 2.0 * math.acos(float(quat[3]))
    return ((quat[:3] * angle) / den).astype(np.float32)

def rotate_180(im): return np.ascontiguousarray(im[::-1, ::-1])


@torch.no_grad()
def predict_action_rollout(raw_obs, task_text, noise_scale=0.0):
    """Действие из baseline policy + опциональный гауссов шум на первые 6 dim
    (НЕ на gripper — иначе ломается логика open/close)."""
    a_img = rotate_180(raw_obs["agentview_image"])
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"])
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    obs = {
        "observation.images.image":  torch.from_numpy(a_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.images.image2": torch.from_numpy(w_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.state":         torch.from_numpy(state).unsqueeze(0).float(),
        "task":                      [task_text],
    }
    obs = preprocessor(obs)
    a = policy.select_action(obs)
    a = postprocessor(a).squeeze(0).detach().cpu().numpy().astype(np.float32)
    if noise_scale > 0:
        # Шум только на первые 6 dim, не на gripper (action[6])
        noise = np.random.normal(0, noise_scale, size=6).astype(np.float32)
        a[:6] = np.clip(a[:6] + noise, -1, 1)
    return a


# ── 3-stage rollout collection: clean → mild noise → strong noise ──
# Идея: разнообразное распределение качества действий → лучшая action-discrimination в Q.
# Чистые rollouts: in-distribution actions, ~75% success.
# Mild noise (σ=0.05): почти as-demo но с микро-перестановками → contrast in critic.
# Strong noise (σ=0.10): больше failures → больше failure-terminals для TD-сигнала.
ROLLOUT_STAGES = [
    {"n": 100, "noise": 0.05, "label": "clean"},
]
N_ROLLOUTS_TOTAL = sum(s["n"] for s in ROLLOUT_STAGES)
ROLLOUT_CACHE_PATH = "/workspace/data/rollout_cache_v0.npz"   # NEW path → старый кэш не подхватится

available_state_ids = [i for i in range(len(init_states)) if i not in INIT_STATES_IDS]
print(f"Available init_states (excl. eval set): {len(available_state_ids)}")
print(f"Total rollouts to collect: {N_ROLLOUTS_TOTAL} ({len(ROLLOUT_STAGES)} stages)")

if os.path.exists(ROLLOUT_CACHE_PATH):
    print(f"Загружаем rollouts из cache {ROLLOUT_CACHE_PATH}")
    data = np.load(ROLLOUT_CACHE_PATH, allow_pickle=True)
    rollout_img1_np    = data["img1"]
    rollout_img2_np    = data["img2"]
    rollout_state_np   = data["state"]
    rollout_action_np  = data["action"]
    rollout_episode_np = data["episode"]
    rollout_frame_np   = data["frame"]
    rollout_episode_success = dict(data["ep_success"].item())
    n_succ = sum(rollout_episode_success.values())
    print(f"  loaded: {len(rollout_action_np)} frames, "
          f"{len(rollout_episode_success)} episodes ({n_succ} success / "
          f"{len(rollout_episode_success)-n_succ} fail)")
else:
    print(f"Сбор {N_ROLLOUTS_TOTAL} rollouts в {len(ROLLOUT_STAGES)} стадии (~1.5-2 часа)…")
    rollout_img1, rollout_img2 = [], []
    rollout_state, rollout_action = [], []
    rollout_episode, rollout_frame = [], []
    rollout_episode_success = {}

    demo_max_ep = int(episode_cache.max())
    next_ep_idx = demo_max_ep + 1
    rollout_count = 0

    t0 = time.time()
    for stage_idx, stage in enumerate(ROLLOUT_STAGES):
        stage_start = time.time()
        stage_succ = 0
        for r in range(stage["n"]):
            state_id = available_state_ids[rollout_count % len(available_state_ids)]
            policy.reset(); env.reset()
            raw_obs = env.set_init_state(init_states[state_id])
            success = False
            traj_frames = []
            for t in range(MAX_STEPS + NUM_STABILIZATION_STEPS):
                if t < NUM_STABILIZATION_STEPS:
                    action = [0.0]*6 + [-1.0]
                else:
                    a = predict_action_rollout(raw_obs, task_description_libero,
                                               noise_scale=stage["noise"])
                    action = a.tolist()
                a_img = rotate_180(raw_obs["agentview_image"]).astype(np.uint8)
                w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"]).astype(np.uint8)
                eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
                eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
                grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
                st = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
                traj_frames.append((a_img, w_img, st, np.array(action[:7], dtype=np.float32)))
                raw_obs, _, done, _ = env.step(action)
                if done:
                    success = True
                    break

            ep_idx = next_ep_idx + rollout_count
            rollout_count += 1
            for f, (im1, im2, st, ac) in enumerate(traj_frames):
                rollout_img1.append(im1); rollout_img2.append(im2)
                rollout_state.append(st); rollout_action.append(ac)
                rollout_episode.append(ep_idx); rollout_frame.append(f)
            rollout_episode_success[ep_idx] = success
            if success: stage_succ += 1

            if (r+1) % 5 == 0 or r == 0:
                elapsed_total = time.time() - t0
                done_total = sum(s["n"] for s in ROLLOUT_STAGES[:stage_idx]) + r + 1
                eta_total = elapsed_total / done_total * (N_ROLLOUTS_TOTAL - done_total)
                sym = '✓' if success else '✗'
                print(f"  [{stage['label']:>13}] {r+1:3d}/{stage['n']} state={state_id:2d} {sym} "
                      f"({len(traj_frames):3d} frames) | stage success: {stage_succ}/{r+1} | "
                      f"total ETA {eta_total/60:.1f}min", flush=True)

        stage_time = time.time() - stage_start
        print(f"  [{stage['label']:>13}] завершён: {stage_succ}/{stage['n']} success "
              f"({stage_succ/stage['n']:.0%}) за {stage_time/60:.1f}min")

    rollout_img1_np    = np.stack(rollout_img1, axis=0).astype(np.uint8)
    rollout_img2_np    = np.stack(rollout_img2, axis=0).astype(np.uint8)
    rollout_state_np   = np.stack(rollout_state, axis=0).astype(np.float32)
    rollout_action_np  = np.stack(rollout_action, axis=0).astype(np.float32)
    rollout_episode_np = np.array(rollout_episode, dtype=np.int64)
    rollout_frame_np   = np.array(rollout_frame, dtype=np.int64)

    n_success_total = sum(rollout_episode_success.values())
    print(f"\n✓ Собрано {N_ROLLOUTS_TOTAL} rollouts: {n_success_total} success / "
          f"{N_ROLLOUTS_TOTAL - n_success_total} fail, total {len(rollout_action_np)} frames")

    os.makedirs(os.path.dirname(ROLLOUT_CACHE_PATH), exist_ok=True)
    np.savez(ROLLOUT_CACHE_PATH,
             img1=rollout_img1_np, img2=rollout_img2_np,
             state=rollout_state_np, action=rollout_action_np,
             episode=rollout_episode_np, frame=rollout_frame_np,
             ep_success=np.array(rollout_episode_success, dtype=object))
    print(f"  сохранено в {ROLLOUT_CACHE_PATH}")


# ── Склейка demo + rollouts ──
combined_img1     = np.concatenate([img1_cache,    rollout_img1_np], axis=0)
combined_img2     = np.concatenate([img2_cache,    rollout_img2_np], axis=0)
combined_state    = np.concatenate([state_cache,   rollout_state_np], axis=0)
combined_action   = np.concatenate([action_cache,  rollout_action_np], axis=0)
combined_episode  = np.concatenate([episode_cache, rollout_episode_np], axis=0)
combined_frame    = np.concatenate([frame_cache,   rollout_frame_np], axis=0)

combined_episode_success = {int(ep): True for ep in np.unique(episode_cache)}
combined_episode_success.update(rollout_episode_success)

combined_state_norm = (combined_state - state_mean) / state_std

n_succ_eps = sum(combined_episode_success.values())
n_fail_eps = len(combined_episode_success) - n_succ_eps
print(f"\nCombined cache:")
print(f"  total frames: {len(combined_action)} ({len(action_cache)} demo + {len(rollout_action_np)} rollouts)")
print(f"  total episodes: {len(combined_episode_success)} ({n_succ_eps} success / {n_fail_eps} fail)")
print(f"  failure ratio: {n_fail_eps / len(combined_episode_success):.1%}")

del policy, preprocessor, postprocessor, env
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 4. Replay buffer

Храним:
- Все images на CPU (uint8, 2.6 GB)
- На каждом sample берём из CPU и переносим на GPU
- Используем pinned memory для скорости

In [ ]:
class ReplayBuffer:
    """Buffer с images на GPU прямо в uint8. ~20 GB на RTX 6000 Ada (48 GB) спокойно."""
    def __init__(self, img1, img2, state_norm, action, reward, done,
                 next_idx, episode_starts, device):
        self.size = len(img1)
        # ── Всё на GPU — один раз, sample затем zero-copy ──
        print(f"  Перенос image cache на GPU…")
        # uint8 на GPU — ~10 GB на cam = 20 GB total
        self.img1 = torch.from_numpy(img1).to(device)        # [N, 256, 256, 3] uint8
        self.img2 = torch.from_numpy(img2).to(device)
        self.state_norm = torch.from_numpy(state_norm).to(device)  # [N, 8] float32
        self.action = torch.from_numpy(action).to(device)
        self.reward = torch.from_numpy(reward).to(device)
        self.done = torch.from_numpy(done).to(device)
        self.next_idx = torch.from_numpy(next_idx).to(device)
        self.device = device
        print(f"  ✓ buffer на GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB used")
    
    def sample(self, batch_size, device):
        # Случайные индексы прямо на GPU — zero-copy fancy indexing
        idx = torch.randint(0, self.size, (batch_size,), device=device)
        next_idx = self.next_idx[idx]
        
        # uint8 → float + permute. Конверсия на GPU быстрая.
        img1c = self.img1[idx].permute(0, 3, 1, 2).float() / 255.0
        img2c = self.img2[idx].permute(0, 3, 1, 2).float() / 255.0
        img1n = self.img1[next_idx].permute(0, 3, 1, 2).float() / 255.0
        img2n = self.img2[next_idx].permute(0, 3, 1, 2).float() / 255.0
        
        return {
            "img1_curr": img1c,
            "img2_curr": img2c,
            "img1_next": img1n,
            "img2_next": img2n,
            "state_curr": self.state_norm[idx],
            "state_next": self.state_norm[next_idx],
            "action": self.action[idx],
            "reward": self.reward[idx],
            "done": self.done[idx],
        }


# Строим transitions из COMBINED кэша (demo + rollouts)
print("Готовим transitions из combined cache…")
N = len(combined_action)

# Sort by (episode, frame)
sort_order = np.lexsort((combined_frame, combined_episode))
img1_sorted       = combined_img1[sort_order]
img2_sorted       = combined_img2[sort_order]
state_norm_sorted = combined_state_norm[sort_order]
action_sorted     = combined_action[sort_order]
episode_sorted    = combined_episode[sort_order]
frame_sorted      = combined_frame[sort_order]

# Boundaries: где меняется episode_index
is_last_in_ep = np.concatenate((episode_sorted[1:] != episode_sorted[:-1], [True]))

# ── Per-episode reward shaping ──
# Successful episode terminal: r = 0   (paper bias = -1, max=0)
# Failed episode terminal:     r = -1  (timeout, no bootstrap)
# Все промежуточные frames:    r = -1
rewards = np.full(N, -1.0, dtype=np.float32)
dones   = is_last_in_ep.astype(np.float32)

# Векторизованная установка terminal-rewards на основе success-mask
last_indices = np.where(is_last_in_ep)[0]
last_episodes = episode_sorted[last_indices]
# Маска успешных эпизодов на позициях last_indices
success_mask = np.array([combined_episode_success.get(int(ep), True) for ep in last_episodes], dtype=bool)
rewards[last_indices[success_mask]] = 0.0  # success terminals → r=0
# Failed terminals остаются -1 (timeout)

# next_idx
next_idx_arr = np.arange(N)
next_idx_arr[~is_last_in_ep] = np.arange(N)[~is_last_in_ep] + 1

episode_starts = np.concatenate(([0], np.where(episode_sorted[1:] != episode_sorted[:-1])[0] + 1))

n_success_terminals = int(success_mask.sum())
n_failure_terminals = int(len(last_indices) - n_success_terminals)
print(f"Total transitions: {N}, episodes: {len(episode_starts)}")
print(f"  ✓ success terminals (r=0):  {n_success_terminals}")
print(f"  ✗ failure terminals (r=-1): {n_failure_terminals}")
print(f"  Avg episode length: {N / len(episode_starts):.1f}")
if n_failure_terminals == 0:
    print("  ⚠ WARNING: failure terminals = 0 — критик не получит action-discriminating signal!")

buffer = ReplayBuffer(
    img1=img1_sorted, img2=img2_sorted, state_norm=state_norm_sorted,
    action=action_sorted, reward=rewards, done=dones,
    next_idx=next_idx_arr, episode_starts=episode_starts,
    device=device,
)
print(f"\nReplay buffer size: {buffer.size}")


## 5. ResNet-18 encoder

Используем torchvision ResNet-18. Из коробки выдаёт 512-d feature после adaptive pool.

Init **with random weights** (`pretrained=False`) — статья говорит "trained from scratch".

In [ ]:
from torchvision.models import resnet18

class ImageEncoder(nn.Module):
    """ResNet-18 from scratch → 512-d feature."""
    def __init__(self, out_dim=512):
        super().__init__()
        self.backbone = resnet18(weights=None)
        # Удалим финальный fc (он 512→1000), оставим adaptive pool
        self.backbone.fc = nn.Identity()
        self.out_dim = 512
    
    def forward(self, img):
        # img: [B, 3, H, W]
        return self.backbone(img)  # [B, 512]


# Sanity test
enc = ImageEncoder().to(device)
test_img = torch.randn(2, 3, 96, 96, device=device)
out = enc(test_img)
print(f"Encoder output shape: {out.shape}")
print(f"Encoder params: {sum(p.numel() for p in enc.parameters())/1e6:.1f}M")
del enc


## 6. Random shift augmentation (DrQ-v2 / PA-RL)

Случайно сдвигаем изображение на ±4 пикселя в каждом направлении. Padding = 4 пикселя по краям, потом random crop. Это улучшает Q-learning стабильность по статье.

In [ ]:
def random_shift_and_downscale(img, target_size=CRITIC_IMG_SIZE, pad=8):
    """
    img: [B, 3, RAW_IMG_SIZE, RAW_IMG_SIZE] float — изображение в полном разрешении 256×256
    Возвращает: [B, 3, target_size, target_size]
    
    Pipeline (DrQ-v2 / PA-RL App B.1):
    1. Pad на ±pad px ⇒ random crop обратно к 256 (shift в raw resolution)
    2. Downscale до target_size (128 — больше стандартных 84 для видимости объектов LIBERO)
    
    pad=8 на raw 256 = эквивалент DrQ-v2 standard pad=4 на final 128
    (4 * 256/128 = 8). Это и есть «4 pixels» из App B.1 в working resolution.
    """
    B, C, H, W = img.shape
    img_padded = F.pad(img, (pad, pad, pad, pad), mode='replicate')
    
    shifts_h = torch.randint(0, 2*pad+1, (B,), device=img.device)
    shifts_w = torch.randint(0, 2*pad+1, (B,), device=img.device)
    
    rows = torch.arange(H, device=img.device).unsqueeze(0) + shifts_h.unsqueeze(1)
    cols = torch.arange(W, device=img.device).unsqueeze(0) + shifts_w.unsqueeze(1)
    
    batch_idx = torch.arange(B, device=img.device).view(B, 1, 1, 1).expand(B, C, H, W)
    chan_idx  = torch.arange(C, device=img.device).view(1, C, 1, 1).expand(B, C, H, W)
    row_idx   = rows.view(B, 1, H, 1).expand(B, C, H, W)
    col_idx   = cols.view(B, 1, 1, W).expand(B, C, H, W)
    
    img_shifted = img_padded[batch_idx, chan_idx, row_idx, col_idx]
    
    if target_size != H:
        img_small = F.interpolate(img_shifted, size=target_size, mode='bilinear', align_corners=False)
    else:
        img_small = img_shifted
    return img_small


def downscale_only(img, target_size=CRITIC_IMG_SIZE):
    """Без shift — для эвала."""
    if target_size != img.shape[-1]:
        return F.interpolate(img, size=target_size, mode='bilinear', align_corners=False)
    return img


# Sanity
test = torch.randn(4, 3, RAW_IMG_SIZE, RAW_IMG_SIZE, device=device)
out = random_shift_and_downscale(test, target_size=CRITIC_IMG_SIZE, pad=8)
print(f"Input:  {test.shape}")
print(f"Output: {out.shape}")
print(f"Pipeline: pad+crop в {RAW_IMG_SIZE} (random shift 8px на raw = эквивалент 4px на {CRITIC_IMG_SIZE}) → downscale до {CRITIC_IMG_SIZE}")


## 7. Twin Q + V networks

Архитектура по статье (Kumar et al. 2023): action concatenates на каждом слое MLP.
Twin Q — два независимых critic'а, target = min(Q1, Q2).

In [ ]:
class CriticHead(nn.Module):
    """Critic head с action embedding — критично для случая когда obs >> action в scale.
    Без этого action [-1,1] заглушается obs из ResNet с большой нормой."""
    def __init__(self, obs_dim, action_dim=7, hidden=512, action_emb_dim=128):
        super().__init__()
        # Action embedding — поднимаем 7-d action до 128-d с LayerNorm
        self.action_emb = nn.Sequential(
            nn.Linear(action_dim, action_emb_dim),
            nn.LayerNorm(action_emb_dim),
            nn.ReLU(),
            nn.Linear(action_emb_dim, action_emb_dim),
            nn.LayerNorm(action_emb_dim),
            nn.ReLU(),
        )
        # Q head: action_emb concatenates на каждом слое (Kumar et al. 2023)
        self.l1 = nn.Sequential(nn.Linear(obs_dim+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l2 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l3 = nn.Sequential(nn.Linear(hidden+action_emb_dim, hidden), nn.LayerNorm(hidden), nn.ReLU())
        self.l4 = nn.Linear(hidden+action_emb_dim, 1)
    def forward(self, obs, a):
        a_emb = self.action_emb(a)  # [B, 128] — нормализованная action representation
        x = self.l1(torch.cat([obs, a_emb], -1))
        x = self.l2(torch.cat([x, a_emb], -1))
        x = self.l3(torch.cat([x, a_emb], -1))
        return self.l4(torch.cat([x, a_emb], -1)).squeeze(-1)


class CriticEnsemble(nn.Module):
    """Image encoders + state MLP + 2 critic heads.
    Енкодеры shared между Q1 и Q2.
    """
    def __init__(self, state_dim=8, action_dim=7, hidden=512, state_hidden=64):
        super().__init__()
        self.enc1 = ImageEncoder()  # overhead camera
        self.enc2 = ImageEncoder()  # wrist camera
        self.state_mlp = nn.Sequential(
            nn.Linear(state_dim, state_hidden), nn.LayerNorm(state_hidden), nn.ReLU(),
            nn.Linear(state_hidden, state_hidden),
        )
        # obs_dim = 512 (img1) + 512 (img2) + state_hidden (64) = 1088
        self.obs_dim = 512 + 512 + state_hidden
        # LayerNorm на финальный obs — критично чтобы obs/action были одного scale
        self.obs_norm = nn.LayerNorm(self.obs_dim)
        self.q1 = CriticHead(self.obs_dim, action_dim, hidden)
        self.q2 = CriticHead(self.obs_dim, action_dim, hidden)
    
    def encode(self, img1, img2, state):
        e1 = self.enc1(img1)                          # [B, 512]
        e2 = self.enc2(img2)                          # [B, 512]
        s  = self.state_mlp(state)                    # [B, 64]
        obs = torch.cat([e1, e2, s], dim=-1)          # [B, 1088]
        # ── LayerNorm — выравнивает scale obs с action_emb ──
        # Без этого obs std ~5-10, action_emb std ~1 → action signal заглушается
        obs = self.obs_norm(obs)
        return obs
    
    def forward(self, img1, img2, state, action):
        obs = self.encode(img1, img2, state)
        return self.q1(obs, action), self.q2(obs, action)
    
    def min_q(self, img1, img2, state, action):
        q1, q2 = self.forward(img1, img2, state, action)
        return torch.minimum(q1, q2)


class VNetwork(nn.Module):
    """V(obs) — share encoder через критика? Нет, отдельный.
    Чтобы избежать доп. проходов encoder'а, можем шарить."""
    def __init__(self, obs_dim, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, obs):
        return self.net(obs).squeeze(-1)


critic = CriticEnsemble().to(device)
critic_target = CriticEnsemble().to(device)
critic_target.load_state_dict(critic.state_dict())
for p in critic_target.parameters():
    p.requires_grad_(False)

v_net = VNetwork(critic.obs_dim).to(device)

print(f"Critic params: {sum(p.numel() for p in critic.parameters())/1e6:.1f}M")
print(f"V-net params: {sum(p.numel() for p in v_net.parameters())/1e6:.1f}M")
print(f"Encoder params per ResNet: {sum(p.numel() for p in critic.enc1.parameters())/1e6:.1f}M")


## 8. IQL + Cal-QL training loop

Параметры по статье (Appendix B.1):
- IQL τ = 0.7 для не-AntMaze
- γ = 0.99
- Cal-QL α = 0.005
- Critic LR = 3e-4
- Soft update τ = 0.005

Random shift augmentation на каждом батче.

In [ ]:
GAMMA = 0.99
TAU_SOFT = 0.005
EXP_TAU = 0.7
CAL_ALPHA = 0.005
CRITIC_LR = 3e-4

critic_optim = optim.Adam(list(critic.parameters()), lr=CRITIC_LR)
v_optim = optim.Adam(v_net.parameters(), lr=CRITIC_LR)

def expectile_loss(u, tau):
    return (torch.abs(tau - (u < 0).float()) * u**2).mean()


def iql_step(batch):
    # Random shift + downscale на GPU
    img1c = random_shift_and_downscale(batch["img1_curr"])
    img2c = random_shift_and_downscale(batch["img2_curr"])
    img1n = random_shift_and_downscale(batch["img1_next"])
    img2n = random_shift_and_downscale(batch["img2_next"])
    state_c = batch["state_curr"]
    state_n = batch["state_next"]
    action = batch["action"]
    reward = batch["reward"]
    done = batch["done"]
    
    # ── ENCODE один раз ──
    # obs_curr — нужен через critic (с градиентом для Q update) и critic_target (для targets)
    # obs_next — только через critic_target (для next_v target)
    
    with torch.no_grad():
        obs_curr_target = critic_target.encode(img1c, img2c, state_c)
        obs_next_target = critic_target.encode(img1n, img2n, state_n)
    obs_curr = critic.encode(img1c, img2c, state_c)  # с градиентом для Q
    
    # ── 1. V update ──
    # V читает obs_curr_target (без градиента в encoder)
    with torch.no_grad():
        q1_target = critic_target.q1(obs_curr_target, action)
        q2_target = critic_target.q2(obs_curr_target, action)
        q_target_val = torch.minimum(q1_target, q2_target)
    v_val = v_net(obs_curr_target)
    v_loss = expectile_loss(q_target_val - v_val, EXP_TAU)
    v_optim.zero_grad(); v_loss.backward(); v_optim.step()
    
    # ── 2. Q update — Bellman + Cal-QL ──
    with torch.no_grad():
        next_v = v_net(obs_next_target)
        bellman_target = reward + GAMMA * (1.0 - done) * next_v
    
    q1 = critic.q1(obs_curr, action)
    q2 = critic.q2(obs_curr, action)
    td_loss = F.mse_loss(q1, bellman_target) + F.mse_loss(q2, bellman_target)
    
    # ── Cal-QL по статье (без margin loss) ──
    # Раньше был margin loss как hack для решения проблемы all-success-data.
    # Теперь buffer содержит mix of success+failure rollouts → TD-сигнал
    # сам по себе несёт action-discrimination, margin loss не нужен.
    a_random = torch.rand_like(action) * 2 - 1
    q1_rand = critic.q1(obs_curr, a_random)
    q2_rand = critic.q2(obs_curr, a_random)
    q_rand = torch.minimum(q1_rand, q2_rand)
    q_demo_avg = (q1 + q2) / 2
    
    # Cal-QL Eq 3.3: max(Q(s,a_pol), V^μ) - Q(s,a_demo)
    # В offline-фазе используем random actions как proxy для policy actions
    with torch.no_grad():
        v_ref = v_net(obs_curr.detach())
    cal_reg = (torch.maximum(q_rand, v_ref) - q_demo_avg.detach()).mean()
    
    # для логирования сохраняем (но не используем в loss-е) margin gap
    margin_loss = F.relu(q_rand - q_demo_avg + 1.0).mean()
    
    q_loss = td_loss + CAL_ALPHA * cal_reg
    critic_optim.zero_grad(); q_loss.backward(); critic_optim.step()
    
    # Soft update target
    with torch.no_grad():
        for p, pt in zip(critic.parameters(), critic_target.parameters()):
            pt.data.mul_(1 - TAU_SOFT).add_(p.data, alpha=TAU_SOFT)
    
    return {
        "td_loss": td_loss.item(),
        "v_loss": v_loss.item(),
        "cal_reg": cal_reg.item(),
        "margin_loss": margin_loss.item(),
        "q_demo": q_demo_avg.mean().item(),
        "q_random": q_rand.mean().item(),
        "v_mean": v_val.mean().item(),
        "q_diff": (q1 - q2).abs().mean().item(),
    }


# Обучение
N_STEPS = 150000  # было 50000: больше времени для сходимости с расширенным buffer-ом
BATCH_SIZE = 256  # с оптимизированным encoder можем себе позволить
LOG_EVERY = 1000

if os.path.exists(CRITIC_PATH):
    print(f"Critic уже сохранён, загружаем")
    state = torch.load(CRITIC_PATH, map_location=device)
    critic.load_state_dict(state["critic"])
    critic_target.load_state_dict(state["critic_target"])
    v_net.load_state_dict(state["v_net"])
    print("OK")
else:
    print(f"Тренируем critic на {N_STEPS} шагах…")
    t0 = time.time()
    for step in range(1, N_STEPS+1):
        batch = buffer.sample(BATCH_SIZE, device)
        m = iql_step(batch)
        if step % LOG_EVERY == 0:
            elapsed = time.time() - t0
            it_s = step / elapsed
            eta_s = (N_STEPS - step) / it_s
            print(f"step {step:6d}/{N_STEPS} | {it_s:5.1f} it/s | ETA {eta_s/60:4.1f}min "
                  f"| td={m['td_loss']:.4f} | margin={m['margin_loss']:.4f} | cal={m['cal_reg']:.4f} "
                  f"| q_demo={m['q_demo']:.4f} | q_rand={m['q_random']:.4f} "
                  f"| q_gap={m['q_demo']-m['q_random']:+.3f} | q_diff={m['q_diff']:.3f}",
                  flush=True)
    
    torch.save({
        "critic": critic.state_dict(),
        "critic_target": critic_target.state_dict(),
        "v_net": v_net.state_dict(),
        "state_mean": state_mean,
        "state_std": state_std,
    }, CRITIC_PATH)
    print(f"Сохранено в {CRITIC_PATH}")

critic.eval()
v_net.eval()


## 9. LIBERO env init

In [ ]:
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy, make_att_2d_masks
from lerobot.policies.factory import make_pre_post_processors

policy = SmolVLAPolicy.from_pretrained(MODEL_ID).to(device).eval()
preprocessor, postprocessor = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=MODEL_ID,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

import builtins
builtins.input = lambda _: "n"
os.environ["LIBERO_DATA_PATH"] = "/workspace/libero_data"
os.makedirs("/workspace/libero_data", exist_ok=True)

from libero.libero import get_libero_path, benchmark
from libero.libero.envs import OffScreenRenderEnv

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict[TASK_SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
task_description_libero = task.language
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)

env = OffScreenRenderEnv(bddl_file_name=task_bddl_file,
                         camera_heights=ENV_IMAGE_SIZE, camera_widths=ENV_IMAGE_SIZE)
env.seed(SEED)
init_states = task_suite.get_task_init_states(TASK_ID)
print(f"Task: {task_description_libero}")
print(f"Init states: {len(init_states)}")


## 10. Q-select эвал helpers

In [ ]:
def quat2axisangle(quat):
    quat = quat.astype(np.float32).copy()
    quat[3] = np.clip(quat[3], -1.0, 1.0)
    den = np.sqrt(max(1e-12, 1.0 - quat[3]**2))
    if np.isclose(den, 0.0): return np.zeros(3, dtype=np.float32)
    angle = 2.0 * math.acos(float(quat[3]))
    return ((quat[:3] * angle) / den).astype(np.float32)

def rotate_180(im): return np.ascontiguousarray(im[::-1, ::-1])


def libero_obs_to_critic_input(raw_obs):
    """raw LIBERO obs (256×256) → (img1, img2, state) для critic'а.
    GPU-side downscale через тот же режим что в обучении."""
    a_img = rotate_180(raw_obs["agentview_image"]).astype(np.uint8)
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"]).astype(np.uint8)
    
    img1_full = torch.from_numpy(a_img).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
    img2_full = torch.from_numpy(w_img).permute(2, 0, 1).unsqueeze(0).float().to(device) / 255.0
    img1_t = downscale_only(img1_full, CRITIC_IMG_SIZE)
    img2_t = downscale_only(img2_full, CRITIC_IMG_SIZE)
    
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    state_norm = (state - state_mean) / state_std
    state_t = torch.from_numpy(state_norm).unsqueeze(0).to(device)
    
    return img1_t, img2_t, state_t


def libero_obs_to_lerobot(raw_obs, task_text):
    """Для policy SmolVLA — нужно полное разрешение и task token."""
    a_img = rotate_180(raw_obs["agentview_image"])
    w_img = rotate_180(raw_obs["robot0_eye_in_hand_image"])
    eef_pos = raw_obs["robot0_eef_pos"].astype(np.float32)
    eef_quat = raw_obs["robot0_eef_quat"].astype(np.float32)
    grip = raw_obs["robot0_gripper_qpos"].astype(np.float32)
    state = np.concatenate([eef_pos, quat2axisangle(eef_quat), grip], 0).astype(np.float32)
    return {
        "observation.images.image":  torch.from_numpy(a_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.images.image2": torch.from_numpy(w_img).permute(2,0,1).unsqueeze(0).float()/255.0,
        "observation.state":         torch.from_numpy(state).unsqueeze(0).float(),
        "task":                      [task_text],
    }


@torch.no_grad()
def sample_chunks_with_q(raw_obs, task_text, n_candidates=8, num_steps_override=4):
    """Семплирует n_candidates chunks от baseline policy + scoring через critic."""
    from lerobot.policies.smolvla.modeling_smolvla import resize_with_pad, pad_vector, make_att_2d_masks
    
    obs = libero_obs_to_lerobot(raw_obs, task_text)
    obs = preprocessor(obs)
    
    img1_pol = obs["observation.images.image"].to(device)
    img2_pol = obs["observation.images.image2"].to(device)
    img1_pol = resize_with_pad(img1_pol, 512, 512, pad_value=0) * 2.0 - 1.0
    img2_pol = resize_with_pad(img2_pol, 512, 512, pad_value=0) * 2.0 - 1.0
    state_pol = pad_vector(obs["observation.state"].to(device), policy.config.max_state_dim)
    lang_tokens = obs["observation.language.tokens"].to(device)
    lang_masks  = obs["observation.language.attention_mask"].to(device)
    mask1 = torch.ones(1, dtype=torch.bool, device=device)
    mask2 = torch.ones(1, dtype=torch.bool, device=device)
    
    # Embed prefix
    prefix_embs, prefix_pad_masks, prefix_att_masks = policy.model.embed_prefix(
        [img1_pol, img2_pol], [mask1, mask2], lang_tokens, lang_masks, state=state_pol
    )
    prefix_att_2d  = make_att_2d_masks(prefix_pad_masks, prefix_att_masks)
    prefix_pos_ids = torch.cumsum(prefix_pad_masks, dim=1) - 1
    _, past_kv = policy.model.vlm_with_expert.forward(
        attention_mask=prefix_att_2d, position_ids=prefix_pos_ids,
        past_key_values=None, inputs_embeds=[prefix_embs, None],
        use_cache=True, fill_kv_cache=True,
    )
    
    actions_shape = (1, policy.config.chunk_size, policy.config.max_action_dim)
    num_steps = num_steps_override or policy.config.num_steps
    dt = -1.0 / num_steps
    
    # Также готовим images для critic'а
    img1_critic, img2_critic, state_critic = libero_obs_to_critic_input(raw_obs)
    
    chunks = []
    q_scores = []
    for _ in range(n_candidates):
        x_t = policy.model.sample_noise(actions_shape, device)
        for step in range(num_steps):
            t_val = 1.0 + step * dt
            t_tensor = torch.tensor(t_val, device=device).expand(1)
            v_t = policy.model.denoise_step(prefix_pad_masks, past_kv, x_t, t_tensor)
            x_t = x_t + dt * v_t
        chunks.append(x_t)
        # Scoring через critic — берём первый action из chunk'а в исходных 7 dim
        first_action = x_t[0, 0, :7].unsqueeze(0)
        q = critic.min_q(img1_critic, img2_critic, state_critic, first_action)
        q_scores.append(q.item())
    
    best_idx = int(np.argmax(q_scores))
    return chunks[best_idx][0], q_scores, best_idx


## 11. Q-select эвал на 13 init_states

In [ ]:
@torch.no_grad()
def predict_q_select(raw_obs, task_text, queue=[]):
    if len(queue) == 0:
        best_chunk, q_scores, best_idx = sample_chunks_with_q(raw_obs, task_text)
        for t in range(policy.config.chunk_size):
            a_norm = best_chunk[t:t+1, :7]
            a_post = postprocessor(a_norm)
            queue.append(a_post.squeeze(0).cpu().numpy().astype(np.float32))
        return queue.pop(0), {"q_scores": q_scores, "best_idx": best_idx}
    return queue.pop(0), None


print("="*60)
print("Q-SELECT eval (ResNet critic)")
print("="*60)
n_success = 0
all_q = []
t0 = time.time()
for state_id in INIT_STATES_IDS:
    policy.reset()
    env.reset()
    raw_obs = env.set_init_state(init_states[state_id])
    queue = []
    success = False
    chunks = 0
    for t in range(MAX_STEPS + NUM_STABILIZATION_STEPS):
        if t < NUM_STABILIZATION_STEPS:
            action = [0.0]*6 + [-1.0]
        else:
            a_np, info = predict_q_select(raw_obs, task_description_libero, queue=queue)
            action = a_np.tolist()
            if info is not None:
                chunks += 1
                all_q.append(info["q_scores"])
        raw_obs, _, done, _ = env.step(action)
        if done:
            success = True
            break
    if success: n_success += 1
    print(f"  state {state_id:3d}: {'✓' if success else '✗'} ({t+1} steps, {chunks} chunks)")

elapsed = time.time() - t0
sr = n_success / len(INIT_STATES_IDS)
print(f"\nQ-SELECT (ResNet critic) SR: {n_success}/{len(INIT_STATES_IDS)} = {sr:.2%}")
print(f"Wall time: {elapsed/60:.1f} min")

if all_q:
    arr = np.array(all_q)
    print(f"\n── Q-scores stats ──")
    print(f"  total chunk-replans: {len(all_q)}")
    print(f"  mean Q:               {arr.mean():.4f}")
    print(f"  std within batch avg: {arr.std(axis=1).mean():.4f}")
    print(f"  best - mean avg:      {(arr.max(axis=1)-arr.mean(axis=1)).mean():.4f}")
    print(f"  best - worst avg:     {(arr.max(axis=1)-arr.min(axis=1)).mean():.4f}")


## 12. Вердикт

In [ ]:
print("="*60)
print("РЕЗУЛЬТАТ ResNet critic vs предыдущие")
print("="*60)
print(f"  Baseline SmolVLA (no critic):     ~7-9/13")
print(f"  Q-select v1 (mean prefix VLM):     8/13 (предыдущие тесты)")
print(f"  Q-select ResNet critic:            {n_success}/13 = {sr:.2%}")
print()
if n_success > 9:
    print("✓ ResNet critic РАБОТАЕТ! Раньше проблема была в эмбеддингах от VLM.")
    print("  → Можно делать distillation policy с этим critic'ом.")
elif n_success >= 8:
    print("○ ResNet critic ≈ baseline. Эмбеддинги были не хуже.")
    print("  → Проблема глубже — возможно сам PA-RL подход не работает на этой задаче.")
else:
    print("⚠ ResNet critic ХУЖЕ baseline. Эмбеддинги не были проблемой.")
    print("  → Принципиальная проблема: critic + 50k frames + sparse reward не дают signal.")
